<img src="./ccsf.png" alt="CCSF Logo" width=200px style="margin:0px -5px">

# Lecture 12: Join

Associated Textbook Sections: [8.4 - 8.5](https://ccsf-math-108.github.io/textbook/chapters/08/4/joining-tables-by-columns/)

---

## Overview

* [Joins](#Joins)
* [Geospatial Data](#Geospatial-Data)

---

## Set Up the Notebook

In [ ]:
from datascience import *
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import warnings
warnings.simplefilter(action='ignore', category=UserWarning)

---

## Joins

---

### Tables, Tables, Tables

- When working with data, rarely do you find that everything you want to know is in one dataset.
- In fact, it can be more efficient to maintain smaller datasets!
- If data is stored in separate places, how do you combine the information?
- For example, here are two tables, `centers` and `instructors`.

In [ ]:
centers = Table.read_table('centers.csv')
centers.show()

In [ ]:
instructors = Table.read_table('instructors.csv')
instructors

- The data in `centers` is unlikely to change as it reflects the physical location of the various CCSF centers and campus.
- The data in `instructors` reflects the instructors at each of the centers teaching MATH 108 and could change every semester.

---

### Joining Two Tables

- Suppose that we want to add MATH 108 instructor details to a map of the centers that offer MATH 108.
- It might be helpful to have the instructor information for the center in one table.

| Center | Address | Latitude | Longitude | Instructor(s) |
|---|---|---|---|---|
| Chinatown/North Beach | 808 Kearny St | 37.7956 | -122.405 | Shawn Wiggins |
| Mission | 1125 Valencia St | 37.7547 | -122.421 | Ron Page |
| Ocean | 50 Frida Kahlo Way | 37.7257 | -122.451 | Sonny Mohammadzadeh |

- A join is the solution for this task, and the table method `join` allows us to do that.

---

### `join`

- The basic anatomy of a `join` call is `tbl1.join(col1, tbl2, col2)`.
- A row of `tbl1` is matched with a row of `tbl2` whenever the value in `col1` equals the value in `col2`.
- Rows with no match in the other table are **dropped**, so the result can have fewer rows than either table.
- The joined column becomes the **first** column of the result, and the rows come out **sorted** by it.
- If a label appears in both tables, the copy from `tbl2` is renamed with a `_2` suffix.
- If the two columns have the same label, the last argument is optional: `tbl1.join(col1, tbl2)`.

<img src="./join_command.svg" alt="Anatomy of a join call, labeling each of the four arguments" width="700px">

---

### Demo: Join

Join `instructors` with `centers` to create a table with all the details from `centers` that shows the instructor(s) for the centers offering MATH 108 this semester.

In [ ]:
math_108 = ...
math_108

---

### Join Details

In the resulting table, where did the rows for John Adams and Evans that were in `centers` go?

<img src="./join_example_table.svg" alt="Joining a drinks table to a discounts table on the cafe name" width="100%">

---

### More Realistic

* In the above example, `instructors` joined with `centers` pretty nicely. 
* A more realistic scenario is to have instructor data stored by CRN (Course Reference Number).
* Instructors in our department are assigned to CRNs, not centers.
* Here is the `sections` data:

In [ ]:
sections = Table.read_table('sections.csv')
sections.show()

---

### Demo: Creating `instructors`

Try to join `sections` with `centers` and notice the issue that comes up.

In [ ]:
...

---

Use `group` to create a table called `instructors` showing a list of instructors for each Location offering MATH 108 this semester.

In [ ]:
...

In [ ]:
...

In [ ]:
# Helper Function
# You are not required to know how this works.
def unique_as_str(collection):
    '''
    Returns the unique items in the collection, joined into a single
    comma-separated string (e.g. 'Ron, Shawn, Sonny' instead of
    ['Ron', 'Shawn', 'Sonny']).

    collection: an array or list of values.
    '''
    uniques = np.unique(np.array(collection).astype(str))
    return ', '.join(uniques)
    
unique_as_str(['Ron', 'Shawn', 'Sonny'])

In [ ]:
instructors = ...
instructors = ...
instructors

---

### Demo: More Details on `join`

What happens when you join tables with the same labels?

In [ ]:
...

* Both tables contribute a column called `Instructor(s)`, so the copy coming from the second table is
renamed `Instructor(s)_2`. 
* Labels in a table always have to be unique.

---

Notice that if the column labels match for the two tables, the last argument is optional.

In [ ]:
...

---

## Geospatial Data

---

### Latitude and Longitude

- **Geospatial data** describes *where* something is, usually as a pair of numbers:
  - **Latitude**: north/south position, from -90 (South Pole) to 90 (North Pole). 
    - San Francisco is about 37.7749.
  - **Longitude**: east/west position, from -180 to 180.
    - San Francisco is about -122.4194.
- **Order matters**: the convention is always **(latitude, longitude)**. 
    - Swapping them is a common bug.
    - San Francisco can be expressed geographically as `(37.7749, -122.4194)`
- A `Table` with geospatial data can lead us to a geographical visualization.
- Here is a table containing the latitude and longitude for San Francisco:

In [ ]:
sf = Table().with_columns(
    'Latitude',  make_array(37.7749),
    'Longitude', make_array(-122.4194))
sf

---

### `maps`

- [`folium`](https://python-visualization.github.io/folium) is a Python library that builds interactive web maps (pan, zoom, click).
- Rather than interacting directly with `folium`, we will use the [`datascience.maps`](https://datascience.readthedocs.io/en/master/maps.html#module-datascience.maps) module to generate a map from a `Table`.
- This module contains `Marker` and `Circle`, which help build a map from a table of latitude and longitude values.
    - `Marker` will generate a map with a pin at each location.
    - `Circle` will generate a map with a circle at each location, where the circle can be customized based on other data values.
- `map_table` reads the **first two columns by position**: latitude first, then longitude.
    - The names of those two columns do not matter. Their *order* does.

---

### Demo: `maps`

* Create a map with a marker at the San Francisco location provided in `sf`.
* Create a map with a circle at the San Francisco location provided in `sf`.

In [ ]:
...

In [ ]:
...

---

### Demo: MATH 108 around CCSF

Create a Table from `math_108` with `'Latitude'` and `'Longitude'` in the first two columns.

Any other column is ignored unless its label is one `map_table` recognizes, such as `labels` or
`area`. Carrying `Center` and `Address` along would not put them on the map, so leave them out
for now.

In [ ]:
# The first two columns have to be the latitudes and longitudes.
# `math_108` starts with 'Center' and 'Address', so this fails.
#Marker.map_table(math_108)

In [ ]:
math_108_geo = math_108.select('Latitude', 'Longitude')
math_108_geo

Create a marker on a map at each campus/center with a section of MATH 108.

In [ ]:
...

---

### Labels and Area

- You can add labels and control circle size by adding columns to the table you pass to `map_table`.
- These columns are found by **name**, not by position, and they can go anywhere after the first two:
    - `labels`: text that pops up when you click the marker or circle. Works for both `Marker` and `Circle`.
    - `area`: the size of a `Circle`, measured in **square pixels**.
- Any column whose label is not one `map_table` recognizes is silently ignored.
- Encoding a count as *area* rather than as radius is the honest choice: doubling the count doubles the ink.
- Watch the scale. The default circle has an area of $\pi \cdot 10^2 \approx 314$ square pixels, so plugging in
  a raw count like `24` gives a circle of radius $\sqrt{24 / \pi} \approx 2.8$ pixels, smaller than the default and
  far too small to compare. Multiply the count by a constant to bring it into a visible range.

---

### Demo: Add Labels

Add the campus/center name and instructor info at each marker.

In [ ]:
...
math_108_for_marker

In [ ]:
...

---

### Demo: Change Area

Update the circle map by making the **area** of the circle represent the number of students registered at the campus/center.

In [ ]:
enrollment = ...
enrollment

In [ ]:
sections_with_enrollment = ...
sections_with_enrollment

In [ ]:
...
enrollment_per_location

In [ ]:
math_108_enrollment = ...
math_108_enrollment

In [ ]:
# Squared pixels per student. Adjust this until the circles are easy to compare.
AREA_PER_STUDENT = 50

def create_labels_for_circles(center, enrollment):
    '''Create a text label from the center and enrollment data.'''
    return 'MATH 108 enrollment at ' + center + ': ' + str(enrollment) + ' students'
    
circle_labels = math_108_enrollment.apply(
    create_labels_for_circles, 'Center', 'Total Enrollment'
)

circle_areas = math_108_enrollment.column('Total Enrollment') * AREA_PER_STUDENT

...
math_108_for_circle

In [ ]:
...

---

## Attribution

This content is licensed under the <a href="https://creativecommons.org/licenses/by-nc-sa/4.0/">Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International License (CC BY-NC-SA 4.0)</a> and derived from the <a href="https://www.data8.org/">Data 8: The Foundations of Data Science</a> offered by the University of California, Berkeley.

<img src="./by-nc-sa.png" width=100px>